For further information on the api look here: https://ranaroussi.github.io/yfinance/reference/yfinance.sector_industry.html$0

In [0]:
%pip install yfinance

import yfinance as yf
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp

In [0]:
# reference spark session
spark = SparkSession.builder.getOrCreate()

In [0]:
# load market information

EUROPE = yf.Market("EUROPE")

EUROPE.summary

In [0]:
aapl = yf.Ticker("AAPL")
tech = yf.Sector(aapl.info.get('sectorKey'))
software = yf.Industry(aapl.info.get('industryKey'))

tech.ticker.info

In [0]:
type(aapl)

In [0]:
import json
from pyspark.sql import Row
from pyspark.sql import functions as F

In [0]:
rows = [
    Row(
        ticker='AAPL',
        source='yahoo_finance',
        raw_json_tech=json.dumps(tech.ticker.info),
        raw_json_software=json.dumps(software.ticker.info)
    )
]

df = spark.createDataFrame(rows)
df.write.mode('overwrite').saveAsTable("bronze.aapl_data")

In [0]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "META", "SPY", "QQQ"]

In [0]:
dfs = []

def fetch_ticker_data(ticker):
    data = yf.download(ticker, period="2y", interval="1d ").reset_index()
    data["Ticker"] = ticker
    data.columns = data.columns.get_level_values(0)
    data["ingestion_time"] = pd.Timestamp.now()
    return data

for ticker in tickers:
    dfs.append(fetch_ticker_data(ticker))

df = pd.concat(dfs, ignore_index=True)


In [0]:
df.head()

In [0]:
spark_df = spark.createDataFrame(df)

In [0]:
# convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
# write to bronze data table
# here the append version is chosen, as common for bronze tables. Further work needs to be done on incremental ingestion
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.stock_time_series")

print(f'Successfully ingested {tickers} stock data to bronze table.')

Incremental Ingestion:

This approach pulls only new or updated data. But yfinance complicates this a bit. We need to identify the difference between the last entry and get the difference added in the form of '2d'. In the future, this will be implemented into the code and the correct form of "append" will be added to the bronze table.